# 05 — Variational Quantum Classifier (VQC)

The VQC composes a ZZ feature map with a `RealAmplitudes` ansatz (2 repetitions, linear
entanglement) and trains 24 parameters with COBYLA to minimise class-weighted binary cross-entropy.
Decisions use `P(qubit 0 = |1>)`, computed from exact statevector probabilities.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def show_image(path):
    """Render a saved PNG inside the notebook without extra display dependencies."""
    img = PILImage.open(path)
    plt.figure(figsize=(min(12, img.width / 100), min(8, img.height / 100)))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


print("Project root:", ROOT)

## 1. Circuit

In [ ]:
circuit_text = (RESULTS / "circuits" / "vqc_circuit.txt").read_text(encoding="utf-8")
print(circuit_text[:3000])

In [ ]:
vres = json.loads((ROOT / "models" / "vqc_meta.json").read_text(encoding="utf-8"))["resources"]
vres

Eight qubits, depth 38, 112 gates, 42 two-qubit gates and 24 trainable parameters. The reported
`shots=1024` is metadata for the finite-shot evaluation path; the main training and test scores use
analytic statevector probabilities.

## 2. Optimisation history

In [ ]:
vmeta = json.loads((ROOT / "models" / "vqc_meta.json").read_text(encoding="utf-8"))
opt = vmeta["info"]["optimizer_result"]
pd.Series(opt).to_frame("COBYLA")

In [ ]:
history = pd.DataFrame(vmeta["info"]["convergence"])
ax = history.plot(x="evaluation", y="loss", legend=False)
ax.set_xlabel("objective evaluation")
ax.set_ylabel("training loss")
ax.set_title("VQC convergence (COBYLA, 150 iterations)")
plt.tight_layout()
plt.show()

The optimiser used all 151 evaluations and stopped at a final loss of 0.4585 without meeting the
convergence criterion. This is a **budget-limited** result, not evidence that the architecture is
incapable; it is reported as such.

## 3. Test performance

In [ ]:
metrics = json.loads((RESULTS / "metrics" / "vqc.json").read_text(encoding="utf-8"))
keys = ["threshold", "accuracy", "precision", "recall", "f1", "roc_auc", "fpr", "fnr", "mcc",
        "balanced_accuracy", "tn", "fp", "fn", "tp"]
pd.Series({k: metrics[k] for k in keys}).to_frame("VQC")

In [ ]:
show_image(RESULTS / "figures" / "vqc_confusion_matrix.png")
show_image(RESULTS / "figures" / "vqc_roc_curve.png")
show_image(RESULTS / "figures" / "vqc_convergence.png")

VQC reaches F1 0.8028 and ROC-AUC 0.8982. It is clearly weaker than both other models on this task,
despite using the same inputs. The high recall (0.9033) comes with a 30.3% false-positive rate.

## 4. Encoding ablation

In [ ]:
enc = pd.read_csv(RESULTS / "tables" / "encoding_comparison.csv")
enc[["model", "encoding", "n_qubits", "circuit_depth", "num_parameters", "f1", "roc_auc", "fpr"]]

In [ ]:
show_image(RESULTS / "figures" / "encoding_comparison.png")

Amplitude encoding uses 3 qubits instead of 8 and performs better here (VQC F1 0.8958 vs 0.8028;
QSVM F1 0.9364 vs 0.9231). That is an empirical encoding result, not evidence of quantum advantage:
for computational-basis state preparation the amplitude kernel reduces to a squared-cosine
(polynomial) kernel, which is classically computable.

## 5. Input sensitivity

In [ ]:
show_image(RESULTS / "figures" / "vqc_feature_sensitivity.png")

**Takeaways**

1. The VQC is implementation-complete (circuit, optimiser history, resources, inference).
2. It did not converge in the 150-iteration budget and is the weakest of the three models.
3. Amplitude encoding is cheaper and stronger on this dataset, but is a classical polynomial
   kernel in disguise.
4. No advantage claim is made.